In [11]:
! pip install retry_requests

  Using cached retry_requests-2.0.0-py3-none-any.whl.metadata (2.6 kB)
Using cached retry_requests-2.0.0-py3-none-any.whl (15 kB)


In [9]:
import pandas as pd
import os
import requests
import urllib.robotparser as rp
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import random as rand
import pandas as pd
import re
import time
import numpy as np
from geopy.geocoders import Nominatim
import openmeteo_requests

import pandas as pd
from retry_requests import retry
import requests_cache


In [2]:
extracted_data_output_path = os.path.join(os.getcwd(), '..', 'data', 'data_extracted')

In [3]:
stadiums = {
    'Arsenal': 'Emirates Stadium',
    'Aston Villa': 'Villa Park',
    'Bournemouth': 'Vitality Stadium',
    'Brentford': 'Gtech Community Stadium',
    'Brighton': 'American Express Community Stadium',
    'Burnley': 'Turf Moor',
    'Cardiff': 'Cardiff City Stadium',
    'Chelsea': 'Stamford Bridge',
    'Crystal Palace': 'Selhurst Park',
    'Everton': 'Goodison Park',
    'Fulham': 'Craven Cottage',
    'Huddersfield': 'John Smith\'s Stadium',
    'Ipswich': 'Portman Road Satdium',
    'Leicester': 'King Power Stadium',
    'Leeds': 'Elland Road Stadium',
    'Liverpool': 'Anfield',
    'Luton': "Kenilworth Road Stadium",
    'Manchester City': "Etihad Stadium",
    'Manchester United': "Old Trafford",
    'Newcastle': "St James' Park",
    'Norwich': "Carrow Road Stadium",
    'Nottingham Forest': "City Ground",
    'Sheffield United': "Bramall Lane",
    'Southampton': "St Mary's Stadium",
    'Stoke': "Bet365 Stadium",
    'Swansea': "Swansea.com Stadium",
    'Tottenham': "Tottenham Hotspur Stadium",
    'Tottenham Old': "Wembley Stadium",
    'Watford': "Vicarage Road Stadium",
    'West Ham': "London Stadium",
    'West Brom': "The Hawthorns",
    "Wolves": "Molineux Stadium",
}


In [4]:

geolocator = Nominatim(user_agent="GetLoc", timeout=10)
stadium_loc_df = pd.DataFrame(columns=['team', 'stadium', 'latitude', 'longitude'])
for team, stadium in stadiums.items():
    location = geolocator.geocode(stadium + ', UK')
    time.sleep(2)  # To avoid hitting the rate limit

    if location is None:
        print(f"Could not find location for {stadium} in {team}.")
        continue
    stadium_loc_df.loc[len(stadium_loc_df)] = [
        team,
        stadium,
        location.latitude,
        location.longitude
    ]
    
stadium_loc_df.loc[len(stadium_loc_df)] = [
    'Sheffield United',
    'Bramall Lane',
    53.369871,
    -1.469003
]
stadium_loc_df.loc[len(stadium_loc_df)] = [
    'Ipswich',
    'Portman Road Satdium',
    52.054509,
    1.146325
]
stadium_loc_df.to_csv(os.path.join(extracted_data_output_path, 'stadiums_location.csv'), index=False)
stadium_loc_df
    

Could not find location for Portman Road Satdium in Ipswich.


,team,stadium,latitude,longitude
0,Arsenal,Emirates Stadium,51.555040,-0.108400
1,Aston Villa,Villa Park,52.509126,-1.885035
2,Bournemouth,Vitality Stadium,50.735181,-1.838328
3,Brentford,Gtech Community Stadium,51.490713,-0.288980
4,Brighton,American Express Community Stadium,50.861547,-0.083693
5,Burnley,Turf Moor,53.789361,-2.229855
6,Cardiff,Cardiff City Stadium,51.472821,-3.202963
7,Chelsea,Stamford Bridge,51.481687,-0.191034
8,Crystal Palace,Selhurst Park,51.398243,-0.085255
9,Everton,Goodison Park,53.438690,-2.966419


In [18]:
stadium_loc_df.iloc[22:31, :]

,team,stadium,latitude,longitude
22,Southampton,St Mary's Stadium,50.905862,-1.390877
23,Stoke,Bet365 Stadium,52.988340,-2.175592
24,Swansea,Swansea.com Stadium,51.642806,-3.934739
25,Tottenham,Tottenham Hotspur Stadium,51.604157,-0.066260
26,Tottenham Old,Wembley Stadium,51.556069,-0.279603
27,Watford,Vicarage Road Stadium,51.649759,-0.401349
28,West Ham,London Stadium,51.538621,-0.016624
29,West Brom,The Hawthorns,52.509165,-1.963662
30,Wolves,Molineux Stadium,52.590272,-2.130408


In [21]:
min_data = '2017-08-11'
max_data = '2025-05-26'

weather_data = None

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
url = "https://archive-api.open-meteo.com/v1/archive"

for index, row in stadium_loc_df.iloc[22:31, :].iterrows():
    
	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	params = {
		"latitude": row['latitude'],
		"longitude": row['longitude'],
		"start_date": min_data,
		"end_date": max_data,
		"hourly": ["temperature_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "surface_pressure", 
             "wind_speed_10m", "relative_humidity_2m", "wind_gusts_10m", "cloud_cover", "et0_fao_evapotranspiration"],
		"timezone": "auto"
	}
	responses = openmeteo.weather_api(url, params=params)

	# Process first location. Add a for-loop for multiple locations or weather models
	response = responses[0]
	print(f"Stadium: {row['stadium']}. Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation {response.Elevation()} m asl")
	print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
	print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	hourly_apparent_temperature = hourly.Variables(1).ValuesAsNumpy()
	hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
	hourly_rain = hourly.Variables(3).ValuesAsNumpy()
	hourly_snowfall = hourly.Variables(4).ValuesAsNumpy()
	hourly_surface_pressure = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_speed_10m = hourly.Variables(6).ValuesAsNumpy()
	hourly_rel_humidity_2m = hourly.Variables(7).ValuesAsNumpy()
	hourly_wind_gusts_10m = hourly.Variables(8).ValuesAsNumpy()
	hourly_cloud_cover = hourly.Variables(9).ValuesAsNumpy()
	hourly_et0_fao_evapotranspiration = hourly.Variables(10).ValuesAsNumpy()
 
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}

	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_data["apparent_temperature"] = hourly_apparent_temperature
	hourly_data["precipitation"] = hourly_precipitation
	hourly_data["rain"] = hourly_rain
	hourly_data["snowfall"] = hourly_snowfall
	hourly_data["surface_pressure"] = hourly_surface_pressure
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["relative_humidity_2m"] = hourly_rel_humidity_2m
	hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
	hourly_data["cloud_cover"] = hourly_cloud_cover
	hourly_data["et0_fao_evapotranspiration"] = hourly_et0_fao_evapotranspiration

	hourly_dataframe = pd.DataFrame(data = hourly_data)
	hourly_dataframe['team'] = row['team']
	hourly_dataframe['stadium'] = row['stadium']
	hourly_dataframe['latitude'] = row['latitude']
	hourly_dataframe['longitude'] = row['longitude']
	if weather_data is None:
		weather_data = hourly_dataframe
	else:
		weather_data = pd.concat([weather_data, hourly_dataframe], ignore_index=True)
	
	time.sleep(30)  # To avoid hitting the rate limit


weather_data

Stadium: St Mary's Stadium. Coordinates: 50.93145751953125°N -1.446441650390625°E
Elevation 12.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Bet365 Stadium. Coordinates: 52.97011947631836°N -2.203369140625°E
Elevation 132.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Swansea.com Stadium. Coordinates: 51.63444519042969°N -3.927276611328125°E
Elevation 24.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Tottenham Hotspur Stadium. Coordinates: 51.63444519042969°N 0.0°E
Elevation 14.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Wembley Stadium. Coordinates: 51.564144134521484°N -0.326690673828125°E
Elevation 46.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Vicarage Road Stadium. Coordinates: 51.63444519042969°N -0.327301025390625°E
Elevation 72.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timez

,date,temperature_2m,apparent_temperature,precipitation,rain,snowfall,surface_pressure,wind_speed_10m,relative_humidity_2m,wind_gusts_10m,cloud_cover,et0_fao_evapotranspiration,team,stadium,latitude,longitude
0,2017-08-10 23:00:00+00:00,12.904000,11.845661,0.0,0.0,0.0,1021.535278,10.233123,90.604660,22.680000,5.0,0.000000,Southampton,St Mary's Stadium,50.905862,-1.390877
1,2017-08-11 00:00:00+00:00,12.454000,11.229650,0.0,0.0,0.0,1021.133240,10.739832,91.479111,18.359999,4.0,0.000000,Southampton,St Mary's Stadium,50.905862,-1.390877
2,2017-08-11 01:00:00+00:00,11.754000,10.680326,0.0,0.0,0.0,1020.830200,9.085988,93.902061,18.719999,1.0,0.000000,Southampton,St Mary's Stadium,50.905862,-1.390877
3,2017-08-11 02:00:00+00:00,11.354000,10.249434,0.0,0.0,0.0,1020.329102,8.788720,94.826698,16.199999,1.0,0.000000,Southampton,St Mary's Stadium,50.905862,-1.390877
4,2017-08-11 03:00:00+00:00,11.154000,9.903081,0.0,0.0,0.0,1020.128052,9.290511,94.502945,16.559999,0.0,0.000000,Southampton,St Mary's Stadium,50.905862,-1.390877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614731,2025-05-26 18:00:00+00:00,12.628500,9.305787,0.1,0.1,0.0,993.855774,22.465412,82.789307,45.360001,100.0,0.065886,Wolves,Molineux Stadium,52.590272,-2.130408
614732,2025-05-26 19:00:00+00:00,12.578500,9.255792,0.1,0.1,0.0,993.360962,22.465412,83.061081,46.439999,100.0,0.054471,Wolves,Molineux Stadium,52.590272,-2.130408
614733,2025-05-26 20:00:00+00:00,12.478500,8.925762,0.2,0.2,0.0,992.666565,24.236582,84.169594,47.880001,100.0,0.046279,Wolves,Molineux Stadium,52.590272,-2.130408
614734,2025-05-26 21:00:00+00:00,12.128500,8.186781,1.9,1.9,0.0,991.564758,27.890127,89.054146,52.919998,100.0,0.032323,Wolves,Molineux Stadium,52.590272,-2.130408


In [13]:
stadiums_weather = pd.read_csv(os.path.join(extracted_data_output_path, 'stadiums_weather.csv'))

In [25]:
weather_data

,date,temperature_2m,apparent_temperature,precipitation,rain,snowfall,surface_pressure,wind_speed_10m,relative_humidity_2m,wind_gusts_10m,cloud_cover,et0_fao_evapotranspiration,team,stadium,latitude,longitude
0,2017-08-10 23:00:00+00:00,10.535001,9.309251,0.0,0.0,0.0,1017.229300,5.771239,87.433130,7.920000,30.0,0.000000,Arsenal,Emirates Stadium,51.555040,-0.108400
1,2017-08-11 00:00:00+00:00,12.285001,11.685767,0.0,0.0,0.0,1016.568540,3.706427,84.430084,7.559999,10.0,0.000000,Arsenal,Emirates Stadium,51.555040,-0.108400
2,2017-08-11 01:00:00+00:00,10.535001,9.614707,0.0,0.0,0.0,1015.936650,5.052841,91.972050,5.760000,30.0,0.000000,Arsenal,Emirates Stadium,51.555040,-0.108400
3,2017-08-11 02:00:00+00:00,10.085000,9.217059,0.0,0.0,0.0,1015.430240,4.693826,94.776010,8.640000,41.0,0.000000,Arsenal,Emirates Stadium,51.555040,-0.108400
4,2017-08-11 03:00:00+00:00,10.285001,9.148931,0.0,0.0,0.0,1015.136050,7.100310,95.422630,11.159999,38.0,0.000000,Arsenal,Emirates Stadium,51.555040,-0.108400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185723,2025-05-26 18:00:00+00:00,12.628500,9.305787,0.1,0.1,0.0,993.855774,22.465412,82.789307,45.360001,100.0,0.065886,Wolves,Molineux Stadium,52.590272,-2.130408
2185724,2025-05-26 19:00:00+00:00,12.578500,9.255792,0.1,0.1,0.0,993.360962,22.465412,83.061081,46.439999,100.0,0.054471,Wolves,Molineux Stadium,52.590272,-2.130408
2185725,2025-05-26 20:00:00+00:00,12.478500,8.925762,0.2,0.2,0.0,992.666565,24.236582,84.169594,47.880001,100.0,0.046279,Wolves,Molineux Stadium,52.590272,-2.130408
2185726,2025-05-26 21:00:00+00:00,12.128500,8.186781,1.9,1.9,0.0,991.564758,27.890127,89.054146,52.919998,100.0,0.032323,Wolves,Molineux Stadium,52.590272,-2.130408


In [27]:
weather_data.to_csv(os.path.join(extracted_data_output_path, 'stadiums_weather.csv'), index=False)